<a href="https://colab.research.google.com/github/aleja71291/FDL-EA-20252/blob/main/7_ConvNet_2Class2_ADNI1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Simplificaremos las clase para crear un modelo binario: sano versus enfermo:

In [ ]:
import os
import zipfile
from google.colab import drive
import pandas as pd
import requests
import shutil

zip_url = 'https://github.com/aleja71291/FDL-EA-20252/raw/refs/heads/main/content/images_Train_2.zip'
csv_url = 'https://raw.githubusercontent.com/aleja71291/FDL-EA-20252/refs/heads/main/content/y_resampled_df%20(1).csv'
zip_file_path = 'data.zip'
csv_file_path = 'labels.csv'
extract_path = 'data/'

response = requests.get(zip_url)
with open(zip_file_path, 'wb') as f:
    f.write(response.content)

response = requests.get(csv_url)
with open(csv_file_path, 'wb') as f:
    f.write(response.content)

with zipfile.ZipFile(zip_file_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

labels_df = pd.read_csv(csv_file_path)

labels_df['simplified_label'] = labels_df['CDGLOBAL'].apply(lambda x: 0 if x in [0, 1] else 1)

labels_df['filename'] = '_' + labels_df.index.astype(str) + '_image.png'

df_to_save = pd.DataFrame()
df_to_save['label'] = labels_df['simplified_label']
df_to_save['filename'] = labels_df['filename']
df_to_save['original_cdglobal'] = labels_df['CDGLOBAL']

df_to_save.to_csv(csv_file_path, index=False)
print("Modified labels.csv with simplified classes has been saved.")

print("\nHead of the new labels.csv:")
print(df_to_save.head())
print("\nSimplified Label Mapping:")
print("Original CDGLOBAL 0, 1 -> New Label 0 (Healthy)")
print("Original CDGLOBAL 2, 3 -> New Label 1 (Diseased)")


Modified labels.csv with simplified classes has been saved.

Head of the new labels.csv:
   label      filename  original_cdglobal
0      0  _0_image.png                  1
1      0  _1_image.png                  0
2      0  _2_image.png                  0
3      0  _3_image.png                  0
4      1  _4_image.png                  2

Simplified Label Mapping:
Original CDGLOBAL 0, 1 -> New Label 0 (Healthy)
Original CDGLOBAL 2, 3 -> New Label 1 (Diseased)


In [ ]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
import pandas as pd
from torchvision import transforms
from PIL import Image
import requests
import zipfile
import shutil

class CustomImageDataset(Dataset):
    def __init__(self, csv_file, root_dir, transform=None):
        self.data_frame = pd.read_csv(csv_file)
        self.root_dir = root_dir
        self.transform = transform

    def __len__(self):
        return len(self.data_frame)

    def __getitem__(self, idx):
        label = int(self.data_frame.iloc[idx, 0])
        img_name = os.path.join(self.root_dir, f'_{idx}_image.png')
        image = Image.open(img_name).convert("L")
        if self.transform:
            image = self.transform(image)
        return image, label

class Net(nn.Module):
    def __init__(self, num_classes):
        super(Net, self).__init__()
        self.conv1 = nn.Conv2d(1, 32, 3, padding=1)
        self.bn1 = nn.BatchNorm2d(32)
        self.conv2 = nn.Conv2d(32, 64, 3, padding=1)
        self.bn2 = nn.BatchNorm2d(64)
        self.conv3 = nn.Conv2d(64, 128, 3, padding=1)
        self.bn3 = nn.BatchNorm2d(128)

        self.fc1 = nn.Linear(128 * 3 * 3, 64)
        self.fc_out = nn.Linear(64, num_classes)
        self.dropout = nn.Dropout(0.5)

    def forward(self, x):
        x = F.relu(self.bn1(self.conv1(x)))
        x = F.max_pool2d(x, 2)
        x = F.relu(self.bn2(self.conv2(x)))
        x = F.max_pool2d(x, 2)
        x = F.relu(self.bn3(self.conv3(x)))
        x = F.max_pool2d(x, 2)
        x = torch.flatten(x, 1)
        x = self.dropout(F.relu(self.fc1(x)))
        x = self.fc_out(x)
        return x


# Data augmentation
train_transform = transforms.Compose([
    transforms.RandomRotation(15),
    transforms.RandomAffine(0, translate=(0.15, 0.15), scale=(0.8, 1.2)),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.RandomResizedCrop(28, scale=(0.8, 1.0)),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,)),
])

val_transform = transforms.Compose([
    transforms.Resize((28, 28)),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,)),
])

full_dataset = CustomImageDataset(csv_file=csv_file_path, root_dir=extract_path, transform=None)
num_classes = len(full_dataset.data_frame.iloc[:, 0].unique())

train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size
train_dataset, val_dataset = random_split(full_dataset, [train_size, val_size])

train_dataset.dataset.transform = train_transform
val_dataset.dataset.transform = val_transform

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = Net(num_classes=num_classes).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)

# Training parameters
num_epochs = 75
best_val_accuracy = 0.0
patience = 10
early_stopping_counter = 0

print(f"Starting training on {device}...")
for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    for i, (images, labels) in enumerate(train_loader):
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * images.size(0)

    epoch_loss = running_loss / len(train_dataset)

    # Validation Loop
    model.eval()
    correct = 0
    total = 0
    val_loss = 0.0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            val_loss += loss.item() * images.size(0)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    epoch_val_loss = val_loss / len(val_dataset)
    val_accuracy = 100 * correct / total

    print(f'Epoch [{epoch+1}/{num_epochs}], '
          f'Train Loss: {epoch_loss:.4f}, '
          f'Validation Loss: {epoch_val_loss:.4f}, '
          f'Validation Accuracy: {val_accuracy:.2f}%')

    # Early stopping logic
    if val_accuracy > best_val_accuracy:
        best_val_accuracy = val_accuracy
        early_stopping_counter = 0
        torch.save(model.state_dict(), 'best_model.pth')
        print(f"  --> Saved best model with accuracy: {best_val_accuracy:.2f}%")
    else:
        early_stopping_counter += 1

    if early_stopping_counter >= patience:
        print("Early stopping triggered.")
        break

print("\nTraining complete!")
print(f"Best validation accuracy achieved: {best_val_accuracy:.2f}%")

# Cleanup
os.remove(zip_file_path)
os.remove(csv_file_path)
shutil.rmtree(extract_path)
print("Cleaned up temporary files.")

Starting training on cpu...
Epoch [1/75], Train Loss: 0.5700, Validation Loss: 0.6178, Validation Accuracy: 64.19%
  --> Saved best model with accuracy: 64.19%
Epoch [2/75], Train Loss: 0.4180, Validation Loss: 0.4538, Validation Accuracy: 79.05%
  --> Saved best model with accuracy: 79.05%
Epoch [3/75], Train Loss: 0.3349, Validation Loss: 0.4551, Validation Accuracy: 84.46%
  --> Saved best model with accuracy: 84.46%
Epoch [4/75], Train Loss: 0.2869, Validation Loss: 0.4627, Validation Accuracy: 78.38%
Epoch [5/75], Train Loss: 0.2245, Validation Loss: 0.4685, Validation Accuracy: 81.08%
Epoch [6/75], Train Loss: 0.2039, Validation Loss: 0.4193, Validation Accuracy: 83.78%
Epoch [7/75], Train Loss: 0.1207, Validation Loss: 0.4110, Validation Accuracy: 83.78%
Epoch [8/75], Train Loss: 0.0646, Validation Loss: 0.6464, Validation Accuracy: 81.08%
Epoch [9/75], Train Loss: 0.0367, Validation Loss: 0.4426, Validation Accuracy: 85.81%
  --> Saved best model with accuracy: 85.81%
Epoch [10

In [ ]:
# Import files from google.colab
from google.colab import files

if os.path.exists('best_model.pth'):
    files.download('best_model.pth')
    print("Downloaded best_model.pth")
else:
    print("Error: best_model.pth not found. Make sure it was saved correctly.")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Downloaded best_model.pth


Ahora

In [ ]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import pandas as pd
from torchvision import transforms
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import accuracy_score, precision_score, recall_score, confusion_matrix
from PIL import Image
import requests
import zipfile
import shutil

zip_url = 'https://github.com/aleja71291/FDL-EA-20252/raw/refs/heads/main/content/images_test_2.zip'
csv_url = 'https://raw.githubusercontent.com/aleja71291/FDL-EA-20252/refs/heads/main/content/y_test%20(1).csv'
zip_file_path = 'test_data.zip'
csv_file_path = 'test_labels.csv'
extract_path = 'test_data/'

print("Downloading test data...")
response = requests.get(zip_url)
with open(zip_file_path, 'wb') as f:
    f.write(response.content)

response = requests.get(csv_url)
with open(csv_file_path, 'wb') as f:
    f.write(response.content)

with zipfile.ZipFile(zip_file_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)
print("Test data downloaded and extracted.")

test_labels_df = pd.read_csv(csv_file_path)

test_labels_df['simplified_label'] = test_labels_df['CDGLOBAL'].apply(lambda x: 0 if x in [0, 1] else 1)

test_labels_df['filename'] = '_' + test_labels_df.index.astype(str) + '_image.png'

df_to_save_test = pd.DataFrame()
df_to_save_test['label'] = test_labels_df['simplified_label']
df_to_save_test['filename'] = test_labels_df['filename']
df_to_save_test['original_cdglobal'] = test_labels_df['CDGLOBAL']

df_to_save_test.to_csv(csv_file_path, index=False)
print("Modified test_labels.csv with simplified classes has been saved.")

class Net(nn.Module):
    def __init__(self, num_classes):
        super(Net, self).__init__()
        self.conv1 = nn.Conv2d(1, 32, 3, padding=1)
        self.bn1 = nn.BatchNorm2d(32)
        self.conv2 = nn.Conv2d(32, 64, 3, padding=1)
        self.bn2 = nn.BatchNorm2d(64)
        self.conv3 = nn.Conv2d(64, 128, 3, padding=1)
        self.bn3 = nn.BatchNorm2d(128)

        self.fc1 = nn.Linear(128 * 3 * 3, 64)
        self.fc_out = nn.Linear(64, num_classes)
        self.dropout = nn.Dropout(0.5)

    def forward(self, x):
        x = F.relu(self.bn1(self.conv1(x)))
        x = F.max_pool2d(x, 2)
        x = F.relu(self.bn2(self.conv2(x)))
        x = F.max_pool2d(x, 2)
        x = F.relu(self.bn3(self.conv3(x)))
        x = F.max_pool2d(x, 2)
        x = torch.flatten(x, 1)
        x = self.dropout(F.relu(self.fc1(x)))
        x = self.fc_out(x)
        return x

class CustomTestImageDataset(Dataset):
    def __init__(self, csv_file, root_dir, transform=None):
        self.data_frame = pd.read_csv(csv_file)
        self.root_dir = root_dir
        self.transform = transform

    def __len__(self):
        return len(self.data_frame)

    def __getitem__(self, idx):
        label = int(self.data_frame.iloc[idx, 0])
        img_name = os.path.join(self.root_dir, self.data_frame.iloc[idx, 1])
        image = Image.open(img_name).convert("L")
        if self.transform:
            image = self.transform(image)
        return image, label

def evaluate(model, data_loader, device):
    model.eval()
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for images, labels in data_loader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            _, preds = torch.max(outputs, 1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    accuracy = accuracy_score(all_labels, all_preds)
    precision = precision_score(all_labels, all_preds, average='weighted', zero_division=0)
    recall = recall_score(all_labels, all_preds, average='weighted', zero_division=0)
    return accuracy, precision, recall, all_labels, all_preds

transform = transforms.Compose([
    transforms.Resize((28, 28)),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,)),
])


test_dataset = CustomTestImageDataset(csv_file=csv_file_path, root_dir=extract_path, transform=transform)
test_data_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)


num_classes_test = 2

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = Net(num_classes=num_classes_test).to(device)
model.load_state_dict(torch.load('best_model.pth'))
model.to(device)

accuracy, precision, recall, all_labels, all_preds = evaluate(model, test_data_loader, device)

print(f'Accuracy: {accuracy:.4f}')
print(f'Precision: {precision:.4f}')
print(f'Recall: {recall:.4f}')

cm = confusion_matrix(all_labels, all_preds)
print(f'\nConfusion Matrix:\n{cm}')

TN, FP, FN, TP = cm.ravel()

print(f'\nTrue Positives (TP): {TP}')
print(f'True Negatives (TN): {TN}')
print(f'False Positives (FP): {FP}')
print(f'False Negatives (FN): {FN}')

specificity = TN / (TN + FP) if (TN + FP) != 0 else 0
print(f'Specificity: {specificity:.4f}')

os.remove(zip_file_path)
shutil.rmtree(extract_path)
os.remove(csv_file_path)
print("Temporary test files removed.")

Test data downloaded and extracted.
Modified test_labels.csv with simplified classes has been saved.
Accuracy: 0.8108
Precision: 0.8179
Recall: 0.8108

Confusion Matrix:
[[70 12]
 [ 9 20]]

True Positives (TP): 20
True Negatives (TN): 70
False Positives (FP): 12
False Negatives (FN): 9
Specificity: 0.8537
Temporary test files removed.
